# Version all-in-one : Min-Variance GAFAM puis Max Sharpe ML

## Objectif pédagogique

Cette section propose une version autonome et reproductible de l'étude en deux étapes :

1. **Portefeuille 1 : Minimum Variance (GAFAM uniquement)**
2. **Portefeuille 2 : Maximum Sharpe piloté par Machine Learning** (GAFAM + actifs refuges `GLD` et `TLT`)

Nous conservons la même discipline empirique :
- séparation stricte **In-Sample (2015-2024)** vs **Out-of-Sample (2025-2026)**,
- estimation robuste du risque via **Ledoit-Wolf**,
- comparaison systématique au benchmark **Nasdaq (`^NDX`)**.

L'objectif est de comprendre pourquoi la minimisation du risque seule peut être insuffisante, puis comment enrichir l'allocation en combinant diversification inter-classes d'actifs et prédiction des rendements attendus (`μ`).

In [ ]:
# ==========================================================
# All-in-one | 1) Imports, paramètres, fonctions utilitaires
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "Le package xgboost est requis. Installation: pip install xgboost"
    ) from exc

plt.style.use("seaborn-v0_8")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Paramètres globaux (Python 3.10+)
TRADING_DAYS: int = 252
ROLLING_WINDOW: int = 60
IN_SAMPLE_END: str = "2024-12-31"
OOS_START: str = "2025-01-01"
OOS_END: str = "2026-12-31"
START_DATE: str = "2015-01-01"
BENCHMARK: str = "^NDX"

GAFAM = ["AAPL", "AMZN", "GOOGL", "META", "MSFT"]
UNIVERSE_EXTENDED = GAFAM + ["GLD", "TLT"]


def download_adj_close(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    """Télécharge les Adj Close avec gestion d'erreur minimale."""
    raw = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
    )
    if raw.empty:
        raise RuntimeError("Données vides via yfinance. Vérifier tickers/période.")
    close = raw["Adj Close"].dropna(how="all")
    if close.empty:
        raise RuntimeError("Adj Close vide après nettoyage.")
    return close


def compute_log_returns(prices: pd.DataFrame | pd.Series) -> pd.DataFrame | pd.Series:
    """Rendements logarithmiques (plus adaptés aux modèles quantitatifs)."""
    return np.log(prices / prices.shift(1)).dropna()


def get_daily_risk_free(start: str, end: str, trading_days: int = 252) -> pd.Series:
    """Construit un taux sans risque journalier à partir de tickers de taux US."""
    rf_tickers = ["^IRX", "^TNX", "^FVX"]
    rf_yield = None

    for ticker in rf_tickers:
        try:
            tmp = yf.download([ticker], start=start, end=end, auto_adjust=False, progress=False)
            if isinstance(tmp.columns, pd.MultiIndex):
                candidate = tmp["Adj Close"][ticker].dropna()
            else:
                candidate = tmp["Adj Close"].dropna()
            if len(candidate) > 0:
                rf_yield = candidate
                break
        except Exception:
            continue

    if rf_yield is None:
        raise RuntimeError(f"Impossible de récupérer le taux sans risque ({rf_tickers}).")

    rf_annual = rf_yield / 100.0
    rf_daily = (1.0 + rf_annual) ** (1.0 / trading_days) - 1.0
    return rf_daily


def max_drawdown_from_logret(logret: pd.Series) -> float:
    """Calcule le drawdown maximum à partir d'une série de log-rendements."""
    wealth = 100 * np.exp(logret.cumsum())
    return float((wealth / wealth.cummax() - 1.0).min())


def jensen_alpha_annualized(port_ret: pd.Series, bench_ret: pd.Series, rf: pd.Series, trading_days: int = 252) -> float:
    """Alpha de Jensen annualisé (cadre CAPM sur rendements excédentaires)."""
    aligned = pd.concat([port_ret, bench_ret, rf], axis=1).dropna()
    aligned.columns = ["rp", "rb", "rf"]

    x = (aligned["rb"] - aligned["rf"]).values
    y = (aligned["rp"] - aligned["rf"]).values

    var_x = np.var(x)
    if var_x <= 1e-12:
        return np.nan

    beta = np.cov(x, y, ddof=1)[0, 1] / var_x
    alpha_daily = np.mean(y) - beta * np.mean(x)
    return float(alpha_daily * trading_days)


def performance_table(port_ret: pd.Series, bench_ret: pd.Series, rf: pd.Series) -> pd.Series:
    """Métriques annualisées: rendement, vol, sharpe, mdd, alpha."""
    aligned = pd.concat([port_ret, bench_ret, rf], axis=1).dropna()
    aligned.columns = ["rp", "rb", "rf"]

    rp = aligned["rp"]
    rb = aligned["rb"]
    rf_series = aligned["rf"]

    ann_return = float(np.exp(rp.mean() * TRADING_DAYS) - 1.0)
    ann_vol = float(rp.std() * np.sqrt(TRADING_DAYS))
    sharpe = float(((rp - rf_series).mean() / rp.std()) * np.sqrt(TRADING_DAYS)) if rp.std() > 0 else np.nan
    mdd = max_drawdown_from_logret(rp)
    alpha = jensen_alpha_annualized(rp, rb, rf_series, TRADING_DAYS)

    return pd.Series(
        {
            "Rendement annuel": ann_return,
            "Volatilité annuelle": ann_vol,
            "Ratio de Sharpe": sharpe,
            "Max Drawdown": mdd,
            "Alpha de Jensen": alpha,
        }
    )


def plot_wealth_curves(curves: dict[str, pd.Series], title: str) -> None:
    """Trace plusieurs courbes de richesse cumulée (base 100)."""
    plt.figure(figsize=(11, 6))
    for label, series in curves.items():
        wealth = 100 * np.exp(series.cumsum())
        plt.plot(wealth.index, wealth, label=label, lw=2)
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Valeur cumulée (base 100)")
    plt.legend()
    plt.tight_layout()
    plt.show()


print("All-in-one initialisé.")
print(f"Fenêtre OOS: {OOS_START} -> {OOS_END}")

## Étape A — Portefeuille 1 : Minimum Variance sur GAFAM

### Idée financière

Le portefeuille **Minimum Variance** cherche la combinaison de poids qui minimise le risque total :

\[
\min_w \; w^\top \Sigma w
\quad \text{s.c.} \quad \sum_i w_i=1, \; w_i \in [0,1]
\]

Cette approche est utile pour construire une allocation défensive, mais elle n'exploite pas explicitement l'information sur les rendements futurs (`μ`).

### Choix méthodologiques

- Univers : uniquement `AAPL`, `AMZN`, `GOOGL`, `META`, `MSFT`
- Risque : covariance **Ledoit-Wolf** sur fenêtre glissante de 60 jours
- Exécution : poids calculés au jour `t`, appliqués au rendement observé en `t+1`
- Évaluation : période **Out-of-Sample 2025-2026** vs Nasdaq

In [ ]:
# =============================================================
# All-in-one | 2) Backtest Min-Variance GAFAM (référence initiale)
# =============================================================


def min_variance_weights(cov_matrix: np.ndarray, bounds: tuple[float, float] = (0.0, 1.0)) -> np.ndarray:
    """Résout numériquement le problème min-variance long-only avec somme des poids = 1."""
    n = cov_matrix.shape[0]
    cov_reg = cov_matrix + 1e-8 * np.eye(n)  # stabilisation numérique

    def objective(w: np.ndarray) -> float:
        return float(w @ cov_reg @ w)

    constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)
    bnds = tuple(bounds for _ in range(n))
    w0 = np.full(n, 1.0 / n)

    try:
        res = minimize(objective, w0, method="SLSQP", bounds=bnds, constraints=constraints)
        if not res.success:
            raise ValueError(res.message)
        w = res.x
    except Exception:
        # Fallback robuste si solveur instable
        w = w0

    w = np.clip(w, bounds[0], bounds[1])
    return w / w.sum() if w.sum() > 0 else w0


# Données GAFAM + benchmark
prices_mv = download_adj_close(GAFAM + [BENCHMARK], START_DATE, OOS_END)
prices_mv_assets = prices_mv[GAFAM].dropna()
prices_mv_bench = prices_mv[BENCHMARK].dropna()

ret_mv_assets = compute_log_returns(prices_mv_assets)
ret_mv_bench = compute_log_returns(prices_mv_bench)

common_mv = ret_mv_assets.index.intersection(ret_mv_bench.index)
ret_mv_assets = ret_mv_assets.loc[common_mv]
ret_mv_bench = ret_mv_bench.loc[common_mv]
rf_mv = get_daily_risk_free(START_DATE, OOS_END, TRADING_DAYS).reindex(common_mv).ffill().bfill()

# Backtest OOS
next_ret_mv_assets = ret_mv_assets.shift(-1)
next_ret_mv_bench = ret_mv_bench.shift(-1)

port_mv, bench_mv, dates_mv, w_hist_mv = [], [], [], []
for t in ret_mv_assets.index:
    if t < pd.Timestamp(OOS_START) or t > pd.Timestamp(OOS_END):
        continue

    window = ret_mv_assets.loc[:t].tail(ROLLING_WINDOW)
    if len(window) < ROLLING_WINDOW:
        continue

    sigma_t = LedoitWolf().fit(window.values).covariance_
    w_t = min_variance_weights(sigma_t)

    r_next = next_ret_mv_assets.loc[t]
    b_next = next_ret_mv_bench.loc[t]
    if r_next.isna().any() or pd.isna(b_next):
        continue

    port_mv.append(float(np.dot(w_t, r_next.values)))
    bench_mv.append(float(b_next))
    dates_mv.append(t)
    w_hist_mv.append(w_t)

ret_port_mv = pd.Series(port_mv, index=dates_mv, name="Min Variance GAFAM")
ret_bench_mv = pd.Series(bench_mv, index=dates_mv, name="Nasdaq")
weights_mv = pd.DataFrame(w_hist_mv, index=dates_mv, columns=GAFAM)

plot_wealth_curves(
    {
        "Portefeuille Min Variance (GAFAM)": ret_port_mv,
        "Nasdaq (^NDX)": ret_bench_mv,
    },
    title="Backtest OOS 2025-2026 | Portefeuille Min Variance GAFAM",
)

metrics_mv = pd.concat(
    [
        performance_table(ret_port_mv, ret_bench_mv, rf_mv.reindex(ret_port_mv.index)).rename("Min Variance GAFAM"),
        performance_table(ret_bench_mv, ret_bench_mv, rf_mv.reindex(ret_port_mv.index)).rename("Nasdaq"),
    ],
    axis=1,
)

metrics_mv_display = metrics_mv.copy()
for metric in ["Rendement annuel", "Volatilité annuelle", "Max Drawdown", "Alpha de Jensen"]:
    metrics_mv_display.loc[metric] = 100 * metrics_mv_display.loc[metric]

print("Métriques OOS | Min Variance GAFAM vs Nasdaq")
display(metrics_mv_display.style.format({"Min Variance GAFAM": "{:.2f}", "Nasdaq": "{:.2f}"}))

print("\nPoids moyens (Min Variance):")
display(weights_mv.mean().sort_values(ascending=False).to_frame("Poids moyen"))

## Étape B — Portefeuille 2 : Maximum Sharpe ML (GAFAM + GLD + TLT)

### Pourquoi élargir l'univers ?

Les actions GAFAM restent fortement corrélées entre elles. L'ajout de l'or (`GLD`) et des obligations longues (`TLT`) améliore la diversification structurelle, surtout en phases de stress de marché.

### Pourquoi passer de Min-Variance à Max-Sharpe ?

Le portefeuille Min-Variance ignore les rendements attendus. Le portefeuille Max-Sharpe, lui, cherche le meilleur compromis rendement/risque :

\[
\max_w \; \frac{w^\top \mu - r_f}{\sqrt{w^\top \Sigma w}}
\]

### Feature Engineering + anti-data-leakage

Pour chaque actif, on construit des variables sur les rendements stationnaires :
- `lag_1`, `lag_3`, `lag_5`,
- moyenne mobile `10j` et `20j`,
- volatilité glissante `20j`.

La cible est le rendement à `t+1`. Pour éviter toute fuite d'information, l'apprentissage est validé via `TimeSeriesSplit` (ordre temporel strict).

In [ ]:
# ====================================================================
# All-in-one | 3) Modèle XGBoost avec TimeSeriesSplit (μ prédits OOS)
# ====================================================================


def build_features_target(ret: pd.Series) -> pd.DataFrame:
    """Features techniques + cible t+1 pour un actif donné."""
    df = pd.DataFrame(index=ret.index)
    df["lag_1"] = ret.shift(1)
    df["lag_3"] = ret.shift(3)
    df["lag_5"] = ret.shift(5)
    df["ma_10"] = ret.rolling(10).mean()
    df["ma_20"] = ret.rolling(20).mean()
    df["vol_20"] = ret.rolling(20).std()
    df["target_t_plus_1"] = ret.shift(-1)
    return df.dropna()


def fit_predict_mu_xgb(asset_name: str, ret: pd.Series, in_sample_end: str, oos_start: str, oos_end: str, n_splits: int = 5):
    """Entraîne XGBoost avec TimeSeriesSplit et prédit μ sur la période OOS."""
    ds = build_features_target(ret)

    train = ds.loc[ds.index <= pd.Timestamp(in_sample_end)].copy()
    oos = ds.loc[(ds.index >= pd.Timestamp(oos_start)) & (ds.index <= pd.Timestamp(oos_end))].copy()

    if len(train) < 200:
        raise RuntimeError(f"Pas assez de données d'entraînement pour {asset_name}.")
    if oos.empty:
        raise RuntimeError(f"Pas de données OOS pour {asset_name}.")

    X_train = train.drop(columns=["target_t_plus_1"])
    y_train = train["target_t_plus_1"]
    X_oos = oos.drop(columns=["target_t_plus_1"])

    # Validation temporelle stricte (évite le look-ahead bias)
    cv = TimeSeriesSplit(n_splits=n_splits)
    rmse_folds = []

    for tr_idx, va_idx in cv.split(X_train):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        model_fold = XGBRegressor(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1,
        )
        model_fold.fit(X_tr, y_tr)
        pred_va = model_fold.predict(X_va)
        rmse_folds.append(np.sqrt(mean_squared_error(y_va, pred_va)))

    # Modèle final sur tout l'In-Sample
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)

    mu_oos = pd.Series(model.predict(X_oos), index=X_oos.index, name=asset_name)
    importance = pd.Series(model.feature_importances_, index=X_train.columns, name=asset_name)

    cv_stats = {
        "asset": asset_name,
        "cv_rmse_mean": float(np.mean(rmse_folds)),
        "cv_rmse_std": float(np.std(rmse_folds)),
    }

    return mu_oos, importance, cv_stats


# Données univers élargi
prices_ml = download_adj_close(UNIVERSE_EXTENDED + [BENCHMARK], START_DATE, OOS_END)
prices_ml_assets = prices_ml[UNIVERSE_EXTENDED].dropna()
prices_ml_bench = prices_ml[BENCHMARK].dropna()

ret_ml_assets = compute_log_returns(prices_ml_assets)
ret_ml_bench = compute_log_returns(prices_ml_bench)
common_ml = ret_ml_assets.index.intersection(ret_ml_bench.index)

ret_ml_assets = ret_ml_assets.loc[common_ml]
ret_ml_bench = ret_ml_bench.loc[common_ml]
rf_ml = get_daily_risk_free(START_DATE, OOS_END, TRADING_DAYS).reindex(common_ml).ffill().bfill()

# Entraînement XGBoost par actif
mu_preds, feature_imps, cv_rows = [], [], []
for asset in UNIVERSE_EXTENDED:
    mu_asset, imp_asset, cv_asset = fit_predict_mu_xgb(
        asset_name=asset,
        ret=ret_ml_assets[asset],
        in_sample_end=IN_SAMPLE_END,
        oos_start=OOS_START,
        oos_end=OOS_END,
        n_splits=5,
    )
    mu_preds.append(mu_asset)
    feature_imps.append(imp_asset)
    cv_rows.append(cv_asset)

mu_pred_oos = pd.concat(mu_preds, axis=1).dropna(how="any")
feature_imp_df = pd.concat(feature_imps, axis=1)
cv_table_ml = pd.DataFrame(cv_rows).set_index("asset").sort_index()

print("Validation TimeSeriesSplit (RMSE)")
display(cv_table_ml)

# Importance moyenne des features (interprétabilité globale)
imp_mean = feature_imp_df.mean(axis=1).sort_values(ascending=True)
plt.figure(figsize=(9, 5))
imp_mean.plot(kind="barh", color="teal")
plt.title("XGBoost | Importance moyenne des features")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# ======================================================================
# All-in-one | 4) Optimisation dynamique Max Sharpe + comparaison finale
# ======================================================================


def max_sharpe_weights(mu: np.ndarray, cov: np.ndarray, rf_daily_value: float, bounds: tuple[float, float] = (0.0, 1.0)) -> np.ndarray:
    """Résout le problème max-Sharpe via minimisation du Sharpe opposé."""
    n = len(mu)
    cov_reg = cov + 1e-8 * np.eye(n)

    def objective(w: np.ndarray) -> float:
        port_ret = float(np.dot(w, mu))
        port_vol = float(np.sqrt(np.clip(w @ cov_reg @ w, 1e-12, None)))
        return -((port_ret - rf_daily_value) / port_vol)

    constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)
    bnds = tuple(bounds for _ in range(n))
    w0 = np.full(n, 1.0 / n)

    try:
        res = minimize(
            objective,
            w0,
            method="SLSQP",
            constraints=constraints,
            bounds=bnds,
            options={"maxiter": 500, "ftol": 1e-12},
        )
        if not res.success:
            raise ValueError(res.message)
        w = res.x
    except Exception:
        # Fallback robuste en cas d'échec numérique
        w = w0

    w = np.clip(w, bounds[0], bounds[1])
    return w / w.sum() if w.sum() > 0 else w0


# Backtest OOS du portefeuille Max Sharpe ML
next_ret_ml_assets = ret_ml_assets.shift(-1)
next_ret_ml_bench = ret_ml_bench.shift(-1)

port_ml, bench_ml, dates_ml, w_hist_ml = [], [], [], []
for t in mu_pred_oos.index:
    hist_window = ret_ml_assets.loc[:t, UNIVERSE_EXTENDED].tail(ROLLING_WINDOW)
    if len(hist_window) < ROLLING_WINDOW:
        continue

    sigma_t = LedoitWolf().fit(hist_window.values).covariance_
    mu_t = mu_pred_oos.loc[t, UNIVERSE_EXTENDED].values
    rf_t = float(rf_ml.loc[t])

    w_t = max_sharpe_weights(mu_t, sigma_t, rf_t)

    r_next = next_ret_ml_assets.loc[t, UNIVERSE_EXTENDED]
    b_next = next_ret_ml_bench.loc[t]
    if r_next.isna().any() or pd.isna(b_next):
        continue

    port_ml.append(float(np.dot(w_t, r_next.values)))
    bench_ml.append(float(b_next))
    dates_ml.append(t)
    w_hist_ml.append(w_t)

ret_port_ml = pd.Series(port_ml, index=dates_ml, name="Max Sharpe ML")
ret_bench_ml = pd.Series(bench_ml, index=dates_ml, name="Nasdaq")
weights_ml = pd.DataFrame(w_hist_ml, index=dates_ml, columns=UNIVERSE_EXTENDED)

# Alignement final des 3 séries pour comparaison juste
aligned_all = pd.concat(
    [ret_port_mv.rename("Min Variance GAFAM"), ret_port_ml.rename("Max Sharpe ML"), ret_bench_ml.rename("Nasdaq")],
    axis=1,
).dropna()

ret_mv_aligned = aligned_all["Min Variance GAFAM"]
ret_ml_aligned = aligned_all["Max Sharpe ML"]
ret_bench_aligned = aligned_all["Nasdaq"]
rf_aligned = rf_ml.reindex(aligned_all.index)

plot_wealth_curves(
    {
        "Min Variance GAFAM": ret_mv_aligned,
        "Max Sharpe ML (GAFAM+GLD+TLT)": ret_ml_aligned,
        "Nasdaq (^NDX)": ret_bench_aligned,
    },
    title="Comparaison OOS 2025-2026 | Min Variance vs Max Sharpe ML vs Nasdaq",
)

metrics_final = pd.concat(
    [
        performance_table(ret_mv_aligned, ret_bench_aligned, rf_aligned).rename("Min Variance GAFAM"),
        performance_table(ret_ml_aligned, ret_bench_aligned, rf_aligned).rename("Max Sharpe ML"),
        performance_table(ret_bench_aligned, ret_bench_aligned, rf_aligned).rename("Nasdaq"),
    ],
    axis=1,
)

metrics_final_display = metrics_final.copy()
for metric in ["Rendement annuel", "Volatilité annuelle", "Max Drawdown", "Alpha de Jensen"]:
    metrics_final_display.loc[metric] = 100 * metrics_final_display.loc[metric]

print("Tableau comparatif final (annualisé) - valeurs en % sauf Sharpe")
display(
    metrics_final_display.style.format(
        {
            "Min Variance GAFAM": "{:.2f}",
            "Max Sharpe ML": "{:.2f}",
            "Nasdaq": "{:.2f}",
        }
    )
)

print("\nPoids moyens Max Sharpe ML:")
display(weights_ml.mean().sort_values(ascending=False).to_frame("Poids moyen"))

## Lecture des résultats

- Si **Max Sharpe ML** surperforme **Min Variance GAFAM**, cela signifie que l'information prédictive (même imparfaite) et la diversification inter-classes d'actifs apportent de la valeur.
- Si l'écart est faible ou négatif, c'est une information utile : cela suggère de revoir les features, l'hyperparamétrage XGBoost, les contraintes de poids, ou le schéma de rebalancement.

### Pistes d'amélioration

1. Ajouter des coûts de transaction et un turnover penalty.
2. Mettre en place une recherche d'hyperparamètres (toujours avec `TimeSeriesSplit`).
3. Tester des contraintes de portefeuille supplémentaires (poids max par classe d'actifs, budget de risque, etc.).